## Create a point layer from the Gym Nodes application

In [2]:
import arcpy
import json
import requests
import tempfile
import os

GYM_NODES_GEOJSON_API_URL = "https://46.225.163.131.nip.io/api/gym-nodes/geojson"

# ! If you want to run the notebook: CHANGE TO YOUR absolute path
CURRENT_WORKING_DIR_PATH = "C:\\Users\\HP ZBook 17 G5\\Documents\\ArcGIS\\Projects\\ModelingOfTransportationNetworksAndSystems_FinalProject"
GEODATABASE_PATH = os.path.join(CURRENT_WORKING_DIR_PATH, "ModelingOfTransportationNetworksAndSystems_FinalProject.gdb")

# Fetch the data from the backend of the application
resp = requests.get(GYM_NODES_GEOJSON_API_URL)
resp.raise_for_status()
geojson_data = resp.json()

with tempfile.NamedTemporaryFile(suffix=".geojson", delete=False, mode="w", encoding="utf-8") as f:
    json.dump(geojson_data, f)
    geojson_path = f.name

try:
    out_fc = os.path.join(GEODATABASE_PATH, "gym_nodes_WGS84")
    arcpy.conversion.JSONToFeatures(geojson_path, out_fc, "POINT")
finally:
    os.unlink(geojson_path)

## Convert the point layer to the correct coordinate system, delete the old layer

In [3]:
WGS84_CODE = 4326
UTM_35N_CODE = 32635

# Create a new layer with the correct projection
arcpy.management.Project(
    in_dataset=os.path.join(GEODATABASE_PATH, "gym_nodes_WGS84"),
    out_dataset=os.path.join(GEODATABASE_PATH, "gym_nodes_UTM35N"),
    out_coor_system=arcpy.SpatialReference(UTM_35N_CODE)
)

# Delete the layer with WGS84 projection, we don't need it
arcpy.management.Delete(os.path.join(GEODATABASE_PATH, "gym_nodes_WGS84"))

<Result 'true'>

## Ask the operator to paste his/hers location coordinates from his/hers preferred map provider

In [4]:
# * Doesn't work well - gives a wrong location, it's better to ask the user/operator to paste his/hers coordinates

# GET_LOCATION_BY_IP_API_URL = "http://ip-api.com/json/"

# response = requests.get(GET_LOCATION_BY_IP_API_URL)
# data = response.json()
# lat, lon = data['lat'], data['lon']

user_location_coordinates = input("Paste your current location coordinates: ")
lat, lon = map(float, user_location_coordinates.split(", "))

fc_name = "user_location_WGS84"
sr = arcpy.SpatialReference(WGS84_CODE)

fc_path = os.path.join(GEODATABASE_PATH, fc_name)
arcpy.management.CreateFeatureclass(GEODATABASE_PATH, fc_name, "POINT", spatial_reference=sr)

with arcpy.da.InsertCursor(fc_path, ["SHAPE@XY"]) as cursor:
    cursor.insertRow([(lon, lat)])

Paste your current location coordinates:  42.69447556297307, 23.334726621813445


## Convert the user's location layer to the correct coordinate system, delete the old layer

In [5]:
arcpy.management.Project(
    in_dataset=os.path.join(GEODATABASE_PATH, "user_location_WGS84"),
    out_dataset=os.path.join(GEODATABASE_PATH, "user_location_UTM35N"),
    out_coor_system=arcpy.SpatialReference(UTM_35N_CODE)
)

# Delete the layer with WGS84 projection, we don't need it
arcpy.management.Delete(os.path.join(GEODATABASE_PATH, "user_location_WGS84"))

<Result 'true'>

## Set up a routable network dataset from OSM road data in a file geodatabase, ready for network analysis

In [6]:
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("network")

osm_roads = os.path.join(GEODATABASE_PATH, "gis_osm_roads")

# A network dataset must live inside a feature dataset, together with its source feature class, and share the same spatial reference.
transport_feature_dataset = os.path.join(GEODATABASE_PATH, "Transport")
roads_transport_feature_dataset = os.path.join(transport_feature_dataset, "roads")
network_dataset_transport_feature_dataset = os.path.join(transport_feature_dataset, "roads_ND")

# 1) Feature dataset in UTM 35N (same as the gym nodes -> costs come out in meters)
if not arcpy.Exists(transport_feature_dataset):
    arcpy.management.CreateFeatureDataset(
        GEODATABASE_PATH, "Transport", arcpy.SpatialReference(UTM_35N_CODE)
    )

# 2) Bring the roads into the feature dataset, reprojected to UTM 35N
if not arcpy.Exists(roads_transport_feature_dataset):
    arcpy.management.Project(osm_roads, roads_transport_feature_dataset, arcpy.SpatialReference(UTM_35N_CODE))

# 3) Create + build the network dataset. CreateNetworkDataset produces a default "Length" cost attribute (meters).
if not arcpy.Exists(network_dataset_transport_feature_dataset):
    arcpy.na.CreateNetworkDataset(
        feature_dataset=transport_feature_dataset,
        out_name="roads_ND",
        source_feature_class_names=["roads"],
        elevation_model="NO_ELEVATION"
    )
    arcpy.na.BuildNetwork(network_dataset_transport_feature_dataset)

print("Network dataset ready:", network_dataset_transport_feature_dataset)

Network dataset ready: C:\Users\HP ZBook 17 G5\Documents\ArcGIS\Projects\ModelingOfTransportationNetworksAndSystems_FinalProject\ModelingOfTransportationNetworksAndSystems_FinalProject.gdb\Transport\roads_ND


## Solves a Closest Facility network analysis: finding the nearest gym to the user's location via the road network

In [7]:
# Inputs
user_location_UTM35N = os.path.join(GEODATABASE_PATH, "user_location_UTM35N")
gym_nodes_UTM35N = os.path.join(GEODATABASE_PATH, "gym_nodes_UTM35N")

# Output route
nearest_gym = os.path.join(GEODATABASE_PATH, "nearest_gym")

# ---- BUILD THE CLOSEST FACILITY LAYER ----
result = arcpy.na.MakeClosestFacilityLayer(
    in_network_dataset=network_dataset_transport_feature_dataset,
    out_network_analysis_layer="ClosestGym",
    impedance_attribute="Length",
    travel_from_to="TRAVEL_TO",
    default_number_facilities_to_find=1
)
closest_facility = result.getOutput(0)

# ---- SUBLAYER NAMES ----
sub_layers = arcpy.na.GetNAClassNames(closest_facility)
incidents_name = sub_layers["Incidents"]
facilities_name = sub_layers["Facilities"]
routes_name = sub_layers["CFRoutes"]

# ---- LOAD INPUTS ----
gym_oid_field = arcpy.Describe(gym_nodes_UTM35N).OIDFieldName

arcpy.na.AddLocations(closest_facility, incidents_name, user_location_UTM35N)
arcpy.na.AddLocations(closest_facility, facilities_name, gym_nodes_UTM35N, field_mappings=f"Name {gym_oid_field} #")

# ---- SOLVE ----
arcpy.na.Solve(closest_facility)

routes_layer = closest_facility.listLayers(routes_name)[0]
facilities_layer = closest_facility.listLayers(facilities_name)[0]
arcpy.management.CopyFeatures(routes_layer, nearest_gym)

fac_id_to_gym_oid = {}
with arcpy.da.SearchCursor(facilities_layer, ["OID@", "Name"]) as cursor:
    for fac_oid, name in cursor:
        fac_id_to_gym_oid[fac_oid] = name

closest_facility_id = None
shortest_length = None
with arcpy.da.SearchCursor(routes_layer, ["FacilityID", "Total_Length"]) as cursor:
    for facility_id, total_length in cursor:
        closest_facility_id = facility_id
        shortest_length = total_length
        break  # number_of_facilities_to_find = 1 -> first row is the nearest

# ---- SELECT THE NEAREST GYM IN THE ORIGINAL LAYER ----
if closest_facility_id is not None:
    gym_oid = int(fac_id_to_gym_oid[closest_facility_id])
    where_clause = f"{gym_oid_field} = {gym_oid}"
    arcpy.management.MakeFeatureLayer(gym_nodes_UTM35N, "gym_nodes_lyr")
    arcpy.management.SelectLayerByAttribute("gym_nodes_lyr", "NEW_SELECTION", where_clause)
    print(f"Nearest gym: {gym_oid_field} = {gym_oid}  (network distance: {shortest_length:.1f} m)")
else:
    print("No route was solved - check that the user_location and gym nodes "
          "snapped to the road network (try a larger search tolerance in AddLocations).")

Nearest gym: OBJECTID = 1  (network distance: 3671.7 m)
